# Train (Kaggle)

Fine-tunes VideoMAE-Base or Video Swin-Tiny on ASL Citizen using Kaggle's free GPU quota.

## Before running

1. **Add Data** → attach the ASL Citizen mirror.
2. **Accelerator** → GPU T4 x2. Changing this restarts the session and clears `/kaggle/working`.
3. **Internet** → On, so the repository can be cloned.
4. To continue a previous run, also attach that notebook's output under **Add Data → Your Work**.

Then **Run All**. Every cell is idempotent: re-running after a disconnect resumes rather than restarting.

Kaggle caps a session at roughly 9-12 hours and grants about 30 GPU hours a week, so a full baseline spans several sessions. Section 7 explains the save step that makes that work.

## 1. Setup

In [ ]:
import os
import subprocess
import sys

import torch

assert os.path.exists("/kaggle/input"), (
    "This is the KAGGLE notebook, but this runtime is not Kaggle.\n"
    "For Google Colab use notebooks/colab/02_train_colab.ipynb instead."
)

REPO_URL = "https://github.com/Adgonzalez2018/ASL-Recognition-Model.git"

# Cloned to /tmp: writable, not part of the saved output, discarded with the
# session. Nothing later has to clean it up.
subprocess.run(["rm", "-rf", "/tmp/asl"], check=True)
subprocess.run(["git", "clone", "-q", REPO_URL, "/tmp/asl"], check=True)
PROJECT = "/tmp/asl/ASL_training"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT, "--no-deps"], check=True
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av"], check=True)

ARTIFACTS = "/kaggle/working/artifacts"
OUTPUTS = "/kaggle/working/outputs"
os.makedirs(OUTPUTS, exist_ok=True)

DATASET_ROOT = None
for attachment in sorted(os.listdir("/kaggle/input")):
    for root, dirs, _files in os.walk(f"/kaggle/input/{attachment}"):
        if "splits" in dirs and "videos" in dirs:
            DATASET_ROOT = root
            break
    if DATASET_ROOT:
        break
assert DATASET_ROOT, "ASL Citizen not attached. Use Add Data in the sidebar."

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU."


def run(script, **options):
    """Run a project script, streaming its output into this cell.

    Built as an argument list rather than a shell string. IPython's ! escape
    expands $VAR but not {VAR}, and reads $NAME.ext as attribute access, both
    of which have already cost this notebook a debugging session. subprocess
    has no such rules.
    """
    command = [sys.executable, "-u", f"{PROJECT}/scripts/{script}"]
    for key, value in options.items():
        flag = "--" + key.replace("_", "-")
        # Identity, not equality: 0 == False in Python, so a
        # membership test silently drops --probe-limit 0 and
        # --num-workers 0.
        if value is True:
            command.append(flag)
        elif value is not None and value is not False:
            command += [flag, str(value)]

    print(" ".join(command) + "\n")
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code:
        print(f"\n[exit {code}]")
    return code


print(
    f"gpu        {torch.cuda.get_device_name(0)} "
    f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)"
)
print(f"project    {PROJECT}")
print(f"dataset    {DATASET_ROOT}")
print(f"artifacts  {ARTIFACTS}")
print(f"outputs    {OUTPUTS}")

## 2. Manifests

Regenerated each session rather than carried between them. `probe_limit=0` skips video probing, the slow part, so this takes about a minute instead of the full audit's 30-60.

The result is byte-identical to the full audit. The check below proves it by comparing against the identity recorded in the committed audit report, so a changed dataset surfaces here rather than after training.

In [ ]:
run(
    "audit_dataset.py",
    dataset_root=DATASET_ROOT,
    output_dir=ARTIFACTS,
    write_manifests=True,
    probe_limit=0,
    expected_classes=2731,
)

In [ ]:
import json

with open(f"{PROJECT}/artifacts/audits/asl_citizen_audit.json") as handle:
    reference = json.load(handle)
with open(f"{ARTIFACTS}/audits/asl_citizen_audit.json") as handle:
    current = json.load(handle)

for name in ("label_map_identity", "manifest_identity"):
    ok = reference[name] == current[name]
    print(f"{'OK  ' if ok else 'DIFF'} {name}  {reference[name]}")

assert reference["manifest_identity"] == current["manifest_identity"], (
    "The manifests differ from the audited dataset. Do not train until the "
    "difference is understood."
)
assert not current["integrity"]["errors"], current["integrity"]["errors"]
print(f"\nintegrity clean, {current['counts']['manifest_records']:,} records")

## 3. Resume from a previous session

Kaggle clears `/kaggle/working` when a session ends. Attaching a previous version's output under **Add Data → Your Work** lets this cell copy its checkpoints back, and training picks up from there.

On a first run it finds nothing and starts fresh.

In [ ]:
import shutil
from pathlib import Path

restored = []
for attachment in sorted(os.listdir("/kaggle/input")):
    previous = Path(f"/kaggle/input/{attachment}/outputs")
    if not previous.is_dir():
        continue
    for checkpoint in previous.rglob("latest.pt"):
        target = Path(OUTPUTS) / checkpoint.parent.relative_to(previous)
        target.mkdir(parents=True, exist_ok=True)
        for item in checkpoint.parent.iterdir():
            shutil.copy2(item, target / item.name)
        restored.append(str(target))

        run_dir = checkpoint.parent.parent
        for record in ("history.json", "run_metadata.json"):
            source = run_dir / record
            if source.exists():
                shutil.copy2(source, Path(OUTPUTS) / run_dir.relative_to(previous) / record)

print("\n".join(f"restored {p}" for p in restored) or "No previous checkpoints. Fresh run.")

## 4. Choose the run

`video_swin_tiny` is 30M parameters against VideoMAE's 88M and far lighter on activations, so it fits a larger batch and trains roughly 3x faster. It is the Phase 5 comparison architecture, so it is a legitimate baseline in its own right, and a sensible first run when GPU quota is limited.

VideoMAE-Base at 16 frames produces 1,568 tokens per clip, which is heavy for a 15 GB T4. Batch 8 runs out of memory; batch 4 with doubled accumulation keeps the effective batch at 32.

Whatever you change, keep **batch x accumulation = 32** so runs stay comparable.

In [ ]:
MODEL = "video_swin_tiny"  # or "videomae_base"

if MODEL == "videomae_base":
    BATCH_SIZE, GRAD_ACCUM = 4, 8
else:
    BATCH_SIZE, GRAD_ACCUM = 8, 4

WORKERS = 4
EPOCHS = 20

EXPERIMENT = f"exp-{MODEL}-baseline"
RUN_NAME = f"{MODEL}-seed42"

print(
    f"{MODEL}: batch {BATCH_SIZE} x {GRAD_ACCUM} accumulation "
    f"= {BATCH_SIZE * GRAD_ACCUM} effective, {EPOCHS} epochs"
)

## 5. Preflight

Measures what the run will cost before spending quota on it. A few minutes.

1. **Epoch time** — multiply by `EPOCHS`. If that exceeds your weekly quota, lower `EPOCHS` or switch architecture now rather than at hour 25.
2. **Peak memory** — under 15% headroom warns. Halve `BATCH_SIZE` and double `GRAD_ACCUM`.
3. **Bottleneck** — decoding is CPU-bound and Kaggle gives ~4 cores. If it says `data loading`, raising `WORKERS` helps more than any model change.

In [ ]:
run(
    "train_preflight.py",
    model_config=f"{PROJECT}/configs/models/{MODEL}.yaml",
    training_config=f"{PROJECT}/configs/training/baseline.yaml",
    artifacts_dir=ARTIFACTS,
    dataset_root=DATASET_ROOT,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    num_workers=WORKERS,
)

## 6. Train

Re-running resumes from the last checkpoint rather than restarting, so this is the cell to re-run after a disconnect.

In [ ]:
run(
    "train.py",
    model_config=f"{PROJECT}/configs/models/{MODEL}.yaml",
    training_config=f"{PROJECT}/configs/training/baseline.yaml",
    artifacts_dir=ARTIFACTS,
    dataset_root=DATASET_ROOT,
    output_root=OUTPUTS,
    experiment=EXPERIMENT,
    run_name=RUN_NAME,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    num_workers=WORKERS,
)

## 7. Progress

In [ ]:
run_dir = f"{OUTPUTS}/{EXPERIMENT}/{RUN_NAME}"

if os.path.exists(f"{run_dir}/history.json"):
    with open(f"{run_dir}/history.json") as handle:
        history = json.load(handle)

    for entry in history:
        line = f"epoch {entry['epoch']:3d}  loss {entry['train_loss']:.4f}"
        for name, value in entry["validation"].items():
            line += f"  {name} {value:.4f}"
        line += f"  ({entry['duration_seconds'] / 60:.0f} min)"
        if entry["non_finite_losses"]:
            line += f"  [{entry['non_finite_losses']} non-finite]"
        print(line)

    with open(f"{run_dir}/run_metadata.json") as handle:
        metadata = json.load(handle)
    print(f"\nrun kind        {metadata['run_kind']}")
    print(f"effective batch {metadata['effective_batch_size']}")
    print(f"precision       {metadata['precision_active']}")
    print(f"gpu             {metadata['environment'].get('gpu')}")
else:
    print("No history yet. Run the training cell.")

## 8. Save, so the next session can continue

**This is what makes resume work.** `/kaggle/working` is discarded when the session ends unless you save.

1. **Save Version → Save & Run All (Commit)**.
2. Next session: **Add Data → Your Work → this notebook's output**.
3. Section 3 finds the checkpoints and training continues.

*Save & Run All* re-executes from the top. Training resumes rather than restarting, but the session clock restarts too.

## Notes

**Quota.** Roughly 30 GPU hours a week. Being cut off mid-epoch is survivable; being cut off with no saved version is not.

**Out of memory.** Halve `BATCH_SIZE` and double `GRAD_ACCUM` in section 4. Changing only the batch size makes the run non-comparable to others.

**The test split is untouched here.** Final test evaluation is a separate, deliberate step after model and threshold selection are fixed.